# Challenge 2: Online Purchase Prediction (Team 11)

**Goal:** predict whether an online shopping session ends in a purchase (`Revenue = True`).

**Primary metric:** F1 score for the positive class (`Revenue = True`). The target is imbalanced, so accuracy alone is misleading.

**Rules followed:** only `challenge2_train.csv` is used for exploration, feature engineering, validation, tuning, and threshold selection. `challenge2_test.csv` is used once, at the end, to write `predictions.csv`. `ID` is never a predictor.

## 1. Setup and data loading

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (GradientBoostingClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score, classification_report,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_validate)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
DATA = Path("data")

train = pd.read_csv(DATA / "challenge2_train.csv")
test = pd.read_csv(DATA / "challenge2_test.csv")
print("train:", train.shape, "| test:", test.shape)
assert train["ID"].is_unique and test["ID"].is_unique
assert not set(train["ID"]) & set(test["ID"])

## 2. Exploratory data analysis

In [ ]:
y = train["Revenue"].astype(bool)
print(y.value_counts().rename({True: "Purchase (True)", False: "No purchase (False)"}))
print(f"Positive rate: {y.mean():.2%}")
print("Missing values in train:", int(train.isna().sum().sum()), "| test:", int(test.isna().sum().sum()))
train.describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
y.value_counts().plot.bar(ax=axes[0], color=["#98A2B3", "#1570EF"])
axes[0].set_title("Target balance"); axes[0].set_xlabel("Revenue"); axes[0].set_ylabel("Sessions")

rate_pv = train.groupby(train["PageValues"] > 0)["Revenue"].mean()
rate_pv.index = ["PageValues = 0", "PageValues > 0"]
rate_pv.plot.bar(ax=axes[1], color="#1570EF")
axes[1].set_title("Purchase rate by PageValues"); axes[1].set_ylabel("Purchase rate")

month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
train.groupby("Month")["Revenue"].mean().reindex([m for m in month_order if m in set(train["Month"])]).plot.bar(
    ax=axes[2], color="#1570EF")
axes[2].set_title("Purchase rate by month"); axes[2].set_ylabel("Purchase rate")
for ax in axes[1:]:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

print(train.groupby("VisitorType")["Revenue"].agg(["count", "mean"]).round(3))

**EDA takeaways**

- 1,431 of 9,247 training sessions (15.48%) end in a purchase. A model that always predicts "no purchase" is 84.5% accurate but has F1 = 0, so positive-class F1 is the right primary metric.
- `PageValues` separates the classes strongly: 56.9% of sessions with `PageValues > 0` purchase, versus 3.6% of sessions with no page value.
- Purchase rates vary by month, from 1.4% in February to 22.4% in October and 24.8% in November. They also vary by visitor type: new visitors purchase in 24.7% of sessions versus 14.0% for returning visitors.
- Page-count and duration features are heavily right-skewed (for example, `ProductRelated_Duration` has a median of 598 s but a maximum of 63,974 s), so they are log-transformed before scaling.
- There are no missing values in either dataset.

## 3. Preprocessing and feature engineering

In [ ]:
CATEGORICAL = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
SKEWED = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
          "ProductRelated", "ProductRelated_Duration", "PageValues"]


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    # Row-wise features only, so nothing is learned from other rows (no leakage).
    out = df.drop(columns=["ID", "Revenue"], errors="ignore").copy()
    out["Weekend"] = out["Weekend"].astype(int)
    out["TotalPages"] = out[["Administrative", "Informational", "ProductRelated"]].sum(axis=1)
    out["TotalDuration"] = out[["Administrative_Duration", "Informational_Duration",
                                "ProductRelated_Duration"]].sum(axis=1)
    out["HasPageValue"] = (out["PageValues"] > 0).astype(int)
    per_page = out["ProductRelated_Duration"] / out["ProductRelated"].replace(0, np.nan)
    out["ProductDurationPerPage"] = per_page.fillna(0)
    out["IsQ4"] = out["Month"].isin(["Oct", "Nov", "Dec"]).astype(int)
    for col in CATEGORICAL:
        out[col] = out[col].astype(str)
    return out


X = engineer(train)
X_test = engineer(test)
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]
LOG_COLS = SKEWED + ["TotalPages", "TotalDuration", "ProductDurationPerPage"]
OTHER_NUMERIC = [c for c in NUMERIC if c not in LOG_COLS]
print("Features:", X.shape[1], "| numeric:", len(NUMERIC), "| categorical:", len(CATEGORICAL))


def make_preprocessor(scale: bool = True) -> ColumnTransformer:
    # Fitted inside each CV fold through the Pipeline, so validation rows never influence it.
    log_steps = [("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))]
    if scale:
        log_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("log_numeric", Pipeline(log_steps), LOG_COLS),
        ("numeric", StandardScaler() if scale else "passthrough", OTHER_NUMERIC),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])